# Tech Challenge Fase 3 — Avaliação e Interpretabilidade

## Objetivo

Avaliar de forma final e explicável o modelo de alfabetização construído na etapa anterior.

Nesta etapa serão comparadas duas especificações de Regressão Logística:

### Modelo atual
- `rede`
- `sigla_uf`
- `regiao`
- `log_populacao`
- `vinculos_ativos_por_1000_habitantes`
- `proporcao_vinculos_estatutarios`

### Modelo enxuto
- `rede`
- `sigla_uf`
- `log_populacao`
- `vinculos_ativos_por_1000_habitantes`
- `proporcao_vinculos_estatutarios`

A comparação verifica se `regiao`, que é derivada de `sigla_uf`, realmente adiciona valor preditivo.

### Estratégia temporal

- **2023** → treinamento
- **2024** → teste temporal final *out-of-time*

O hiperparâmetro já foi selecionado no Notebook 04:

`regParam = 0.0`

O foco agora é avaliação, estabilidade e interpretação.


## 1. Configurações


In [ ]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import numpy as np
import pandas as pd

CATALOG = "workspace"
GOLD_SCHEMA = "alfabetizacao_gold"

MODEL_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.base_modelagem_aluno"
)

EVALUATION_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.avaliacao_modelo_fase3"
)

COEFFICIENTS_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.coeficientes_modelo_fase3"
)

SEED = 42
BEST_REG_PARAM = 0.0

print("Base:", MODEL_TABLE)
print("regParam final:", BEST_REG_PARAM)


## 2. Leitura da Gold e feature engineering


In [ ]:
df_raw = spark.table(MODEL_TABLE)

regiao_map = F.create_map(
    *[
        F.lit(x)
        for x in [
            "AC", "Norte",
            "AP", "Norte",
            "AM", "Norte",
            "PA", "Norte",
            "RO", "Norte",
            "RR", "Norte",
            "TO", "Norte",

            "AL", "Nordeste",
            "BA", "Nordeste",
            "CE", "Nordeste",
            "MA", "Nordeste",
            "PB", "Nordeste",
            "PE", "Nordeste",
            "PI", "Nordeste",
            "RN", "Nordeste",
            "SE", "Nordeste",

            "DF", "Centro-Oeste",
            "GO", "Centro-Oeste",
            "MT", "Centro-Oeste",
            "MS", "Centro-Oeste",

            "ES", "Sudeste",
            "MG", "Sudeste",
            "RJ", "Sudeste",
            "SP", "Sudeste",

            "PR", "Sul",
            "RS", "Sul",
            "SC", "Sul",
        ]
    ]
)

df = (
    df_raw
    .withColumn(
        "rede",
        F.col("rede").cast("string"),
    )
    .withColumn(
        "regiao",
        regiao_map[F.col("sigla_uf")],
    )
    .withColumn(
        "log_populacao",
        F.log1p(F.col("populacao").cast("double")),
    )
    .withColumn(
        "proporcao_vinculos_estatutarios",
        F.when(
            F.col("quantidade_vinculos_ativos") > 0,
            F.col("quantidade_vinculos_estatutarios")
            / F.col("quantidade_vinculos_ativos"),
        ).otherwise(F.lit(0.0)),
    )
)

df_train = (
    df
    .filter(
        F.col("grupo_modelagem")
        == "DESENVOLVIMENTO_2023"
    )
)

df_test = (
    df
    .filter(
        F.col("grupo_modelagem")
        == "TESTE_TEMPORAL_2024"
    )
)

print("Treino 2023:", df_train.count())
print("Teste 2024:", df_test.count())


## 3. UFs vistas e não vistas no treinamento

Como 2023 possui menos UFs que 2024, é importante identificar quais UFs aparecem apenas no teste temporal.


In [ ]:
ufs_train = {
    row["sigla_uf"]
    for row in (
        df_train
        .select("sigla_uf")
        .distinct()
        .collect()
    )
}

ufs_test = {
    row["sigla_uf"]
    for row in (
        df_test
        .select("sigla_uf")
        .distinct()
        .collect()
    )
}

ufs_nao_vistas = sorted(
    list(ufs_test - ufs_train)
)

print("UFs vistas em 2023:", sorted(ufs_train))
print("UFs não vistas no treino:", ufs_nao_vistas)

display(
    df_test
    .withColumn(
        "uf_nao_vista_treino",
        F.col("sigla_uf").isin(ufs_nao_vistas),
    )
    .groupBy("uf_nao_vista_treino")
    .agg(
        F.count("*").alias("alunos"),
        F.round(
            F.avg("target_alfabetizado") * 100,
            2,
        ).alias("taxa_alfabetizacao"),
    )
)


## 4. Função de construção dos modelos

Os dois modelos utilizam exatamente o mesmo preprocessing e a mesma Regressão Logística.

A única diferença é a presença ou ausência de `regiao`.


In [ ]:
numeric_cols = [
    "log_populacao",
    "vinculos_ativos_por_1000_habitantes",
    "proporcao_vinculos_estatutarios",
]


def build_pipeline(include_regiao=True):
    categorical_cols = [
        "rede",
        "sigla_uf",
    ]

    if include_regiao:
        categorical_cols.append("regiao")

    indexers = [
        StringIndexer(
            inputCol=coluna,
            outputCol=f"{coluna}_idx",
            handleInvalid="keep",
        )
        for coluna in categorical_cols
    ]

    encoder = OneHotEncoder(
        inputCols=[
            f"{c}_idx"
            for c in categorical_cols
        ],
        outputCols=[
            f"{c}_ohe"
            for c in categorical_cols
        ],
        handleInvalid="keep",
    )

    numeric_assembler = VectorAssembler(
        inputCols=numeric_cols,
        outputCol="numeric_features",
    )

    numeric_scaler = StandardScaler(
        inputCol="numeric_features",
        outputCol="numeric_scaled",
        withStd=True,
        withMean=False,
    )

    final_inputs = [
        "rede_ohe",
        "sigla_uf_ohe",
    ]

    if include_regiao:
        final_inputs.append("regiao_ohe")

    final_inputs.append("numeric_scaled")

    final_assembler = VectorAssembler(
        inputCols=final_inputs,
        outputCol="features",
    )

    lr = LogisticRegression(
        labelCol="target_alfabetizado",
        featuresCol="features",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        maxIter=50,
        elasticNetParam=0.0,
        regParam=BEST_REG_PARAM,
        standardization=False,
    )

    return Pipeline(
        stages=[
            *indexers,
            encoder,
            numeric_assembler,
            numeric_scaler,
            final_assembler,
            lr,
        ]
    )


## 5. Treinamento dos dois modelos

Ambos são treinados em todo o ano de 2023.


In [ ]:
pipeline_atual = build_pipeline(
    include_regiao=True
)

pipeline_enxuto = build_pipeline(
    include_regiao=False
)

model_atual = (
    pipeline_atual
    .fit(df_train)
)

model_enxuto = (
    pipeline_enxuto
    .fit(df_train)
)

print("Modelo atual treinado.")
print("Modelo enxuto treinado.")


## 6. Predições no teste temporal de 2024


In [ ]:
pred_atual = (
    model_atual
    .transform(df_test)
    .withColumn(
        "modelo",
        F.lit("logistica_com_regiao"),
    )
)

pred_enxuto = (
    model_enxuto
    .transform(df_test)
    .withColumn(
        "modelo",
        F.lit("logistica_sem_regiao"),
    )
)

print("Predições concluídas.")


## 7. Função de métricas

Além das métricas já usadas no Notebook 04, serão calculadas:

- **specificity / recall da classe 0**
- **balanced accuracy**
- **F1 da classe 0**
- **macro-F1**

Essas métricas evitam depender apenas da classe positiva.


In [ ]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="target_alfabetizado",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)


def calculate_metrics(pred_df, model_name):
    row = (
        pred_df
        .agg(
            F.sum(
                F.when(
                    (F.col("target_alfabetizado") == 1)
                    & (F.col("prediction") == 1),
                    1,
                ).otherwise(0)
            ).alias("tp"),
            F.sum(
                F.when(
                    (F.col("target_alfabetizado") == 0)
                    & (F.col("prediction") == 0),
                    1,
                ).otherwise(0)
            ).alias("tn"),
            F.sum(
                F.when(
                    (F.col("target_alfabetizado") == 0)
                    & (F.col("prediction") == 1),
                    1,
                ).otherwise(0)
            ).alias("fp"),
            F.sum(
                F.when(
                    (F.col("target_alfabetizado") == 1)
                    & (F.col("prediction") == 0),
                    1,
                ).otherwise(0)
            ).alias("fn"),
        )
        .first()
    )

    tp = int(row["tp"])
    tn = int(row["tn"])
    fp = int(row["fp"])
    fn = int(row["fn"])

    total = tp + tn + fp + fn

    accuracy = (
        (tp + tn) / total
        if total
        else 0.0
    )

    precision_1 = (
        tp / (tp + fp)
        if (tp + fp)
        else 0.0
    )

    recall_1 = (
        tp / (tp + fn)
        if (tp + fn)
        else 0.0
    )

    f1_1 = (
        2 * precision_1 * recall_1
        / (precision_1 + recall_1)
        if (precision_1 + recall_1)
        else 0.0
    )

    precision_0 = (
        tn / (tn + fn)
        if (tn + fn)
        else 0.0
    )

    recall_0 = (
        tn / (tn + fp)
        if (tn + fp)
        else 0.0
    )

    f1_0 = (
        2 * precision_0 * recall_0
        / (precision_0 + recall_0)
        if (precision_0 + recall_0)
        else 0.0
    )

    balanced_accuracy = (
        (recall_1 + recall_0) / 2
    )

    macro_f1 = (
        (f1_1 + f1_0) / 2
    )

    roc_auc = (
        auc_evaluator
        .evaluate(pred_df)
    )

    return {
        "modelo": model_name,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision_1,
        "recall": recall_1,
        "specificity": recall_0,
        "f1": f1_1,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "roc_auc": roc_auc,
    }


## 8. Comparação final dos modelos


In [ ]:
metrics_atual = calculate_metrics(
    pred_atual,
    "logistica_com_regiao",
)

metrics_enxuto = calculate_metrics(
    pred_enxuto,
    "logistica_sem_regiao",
)

metric_rows = []

for result in [
    metrics_atual,
    metrics_enxuto,
]:
    for metrica in [
        "accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy",
        "macro_f1",
        "roc_auc",
    ]:
        metric_rows.append(
            (
                result["modelo"],
                metrica,
                float(result[metrica]),
            )
        )

df_model_comparison = (
    spark.createDataFrame(
        metric_rows,
        [
            "modelo",
            "metrica",
            "valor",
        ],
    )
)

display(
    df_model_comparison
    .orderBy(
        "metrica",
        F.desc("valor"),
    )
)


## 9. Matriz de confusão dos dois modelos


In [ ]:
display(
    spark.createDataFrame(
        [
            (
                metrics_atual["modelo"],
                metrics_atual["tp"],
                metrics_atual["tn"],
                metrics_atual["fp"],
                metrics_atual["fn"],
            ),
            (
                metrics_enxuto["modelo"],
                metrics_enxuto["tp"],
                metrics_enxuto["tn"],
                metrics_enxuto["fp"],
                metrics_enxuto["fn"],
            ),
        ],
        [
            "modelo",
            "tp",
            "tn",
            "fp",
            "fn",
        ],
    )
)


## 10. Comparação nas UFs não vistas em 2023

Essa é a análise mais importante para decidir se `regiao` deve permanecer.

Se o modelo com região performar melhor em SP, DF e AC sem prejudicar significativamente o desempenho global, existe justificativa para manter essa feature.


In [ ]:
pred_atual_unseen = (
    pred_atual
    .filter(
        F.col("sigla_uf").isin(
            ufs_nao_vistas
        )
    )
)

pred_enxuto_unseen = (
    pred_enxuto
    .filter(
        F.col("sigla_uf").isin(
            ufs_nao_vistas
        )
    )
)

unseen_atual = calculate_metrics(
    pred_atual_unseen,
    "com_regiao_ufs_nao_vistas",
)

unseen_enxuto = calculate_metrics(
    pred_enxuto_unseen,
    "sem_regiao_ufs_nao_vistas",
)

display(
    spark.createDataFrame(
        [
            (
                unseen_atual["modelo"],
                unseen_atual["accuracy"],
                unseen_atual["balanced_accuracy"],
                unseen_atual["macro_f1"],
                unseen_atual["roc_auc"],
            ),
            (
                unseen_enxuto["modelo"],
                unseen_enxuto["accuracy"],
                unseen_enxuto["balanced_accuracy"],
                unseen_enxuto["macro_f1"],
                unseen_enxuto["roc_auc"],
            ),
        ],
        [
            "modelo",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "roc_auc",
        ],
    )
)


## 11. Seleção automática do modelo final

A regra será simples:

1. priorizar maior ROC-AUC global;
2. se a diferença global for muito pequena, observar balanced accuracy;
3. usar a análise das UFs não vistas como critério de desempate.

A seleção continua sendo explicável e sem nova busca de hiperparâmetros.


In [ ]:
auc_atual = metrics_atual["roc_auc"]
auc_enxuto = metrics_enxuto["roc_auc"]

if auc_enxuto > auc_atual + 0.002:
    FINAL_MODEL_NAME = "logistica_sem_regiao"
    final_model = model_enxuto
    final_pred = pred_enxuto

elif auc_atual > auc_enxuto + 0.002:
    FINAL_MODEL_NAME = "logistica_com_regiao"
    final_model = model_atual
    final_pred = pred_atual

else:
    # Diferença praticamente irrelevante:
    # prioriza balanced accuracy.
    if (
        metrics_enxuto["balanced_accuracy"]
        >= metrics_atual["balanced_accuracy"]
    ):
        FINAL_MODEL_NAME = "logistica_sem_regiao"
        final_model = model_enxuto
        final_pred = pred_enxuto
    else:
        FINAL_MODEL_NAME = "logistica_com_regiao"
        final_model = model_atual
        final_pred = pred_atual

print("Modelo final selecionado:", FINAL_MODEL_NAME)


## 12. Métricas do modelo final por UF

Essa visão identifica onde o modelo generaliza melhor ou pior territorialmente.

Para evitar interpretação baseada em poucos registros, todas as UFs são apresentadas com seu volume de alunos.


In [ ]:
uf_confusion = (
    final_pred
    .groupBy(
        "sigla_uf",
    )
    .agg(
        F.count("*").alias("alunos"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("tp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("tn"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 0)
                & (F.col("prediction") == 1),
                1,
            ).otherwise(0)
        ).alias("fp"),
        F.sum(
            F.when(
                (F.col("target_alfabetizado") == 1)
                & (F.col("prediction") == 0),
                1,
            ).otherwise(0)
        ).alias("fn"),
    )
    .withColumn(
        "accuracy",
        (F.col("tp") + F.col("tn"))
        / F.col("alunos"),
    )
    .withColumn(
        "recall_1",
        F.when(
            (F.col("tp") + F.col("fn")) > 0,
            F.col("tp")
            / (F.col("tp") + F.col("fn")),
        ),
    )
    .withColumn(
        "recall_0",
        F.when(
            (F.col("tn") + F.col("fp")) > 0,
            F.col("tn")
            / (F.col("tn") + F.col("fp")),
        ),
    )
    .withColumn(
        "balanced_accuracy",
        (
            F.col("recall_1")
            + F.col("recall_0")
        ) / 2,
    )
)

display(
    uf_confusion
    .select(
        "sigla_uf",
        "alunos",
        F.round("accuracy", 4).alias("accuracy"),
        F.round(
            "balanced_accuracy",
            4,
        ).alias("balanced_accuracy"),
        F.round("recall_1", 4).alias("recall_alfabetizado"),
        F.round("recall_0", 4).alias("recall_nao_alfabetizado"),
    )
    .orderBy(
        F.desc("balanced_accuracy")
    )
)


## 13. Métricas por região


In [ ]:
display(
    final_pred
    .groupBy("regiao")
    .agg(
        F.count("*").alias("alunos"),
        F.avg(
            F.when(
                F.col("prediction")
                == F.col("target_alfabetizado"),
                1.0,
            ).otherwise(0.0)
        ).alias("accuracy"),
        F.avg(
            "target_alfabetizado"
        ).alias("taxa_real_alfabetizacao"),
        F.avg(
            "prediction"
        ).alias("taxa_predita_alfabetizacao"),
    )
    .select(
        "regiao",
        "alunos",
        F.round("accuracy", 4).alias("accuracy"),
        F.round(
            F.col("taxa_real_alfabetizacao") * 100,
            2,
        ).alias("taxa_real_alfabetizacao"),
        F.round(
            F.col("taxa_predita_alfabetizacao") * 100,
            2,
        ).alias("taxa_predita_alfabetizacao"),
    )
    .orderBy(
        F.desc("accuracy")
    )
)


## 14. Análise dos erros

Os erros são classificados em:

- **FP**: previsto alfabetizado, mas não alfabetizado;
- **FN**: previsto não alfabetizado, mas alfabetizado;
- **ACERTO**: classificação correta.


In [ ]:
pred_erros = (
    final_pred
    .withColumn(
        "tipo_resultado",
        F.when(
            F.col("prediction")
            == F.col("target_alfabetizado"),
            F.lit("ACERTO"),
        )
        .when(
            (F.col("prediction") == 1)
            & (F.col("target_alfabetizado") == 0),
            F.lit("FP"),
        )
        .otherwise(
            F.lit("FN"),
        ),
    )
)

display(
    pred_erros
    .groupBy(
        "tipo_resultado",
    )
    .agg(
        F.count("*").alias("alunos"),
        F.round(
            F.avg("populacao"),
            2,
        ).alias("populacao_media"),
        F.round(
            F.avg(
                "vinculos_ativos_por_1000_habitantes"
            ),
            2,
        ).alias("vinculos_por_1000_media"),
        F.round(
            F.avg(
                "proporcao_vinculos_estatutarios"
            ),
            4,
        ).alias("proporcao_estatutarios_media"),
    )
    .orderBy(
        F.desc("alunos")
    )
)


## 15. Função para extrair coeficientes e odds ratios

Na Regressão Logística:

`odds_ratio = exp(coeficiente)`

Interpretação:

- `odds_ratio > 1` → associação positiva;
- `odds_ratio < 1` → associação negativa;
- `odds_ratio ≈ 1` → efeito pequeno.

A interpretação é **associativa, não causal**.


In [ ]:
def extract_coefficients(model, sample_df):
    lr_model = model.stages[-1]

    coefs = (
        lr_model
        .coefficients
        .toArray()
    )

    transformed = (
        model
        .transform(
            sample_df.limit(1)
        )
    )

    metadata = (
        transformed
        .schema["features"]
        .metadata
    )

    attrs = (
        metadata["ml_attr"]["attrs"]
    )

    flattened = []

    for attr_type in attrs:
        flattened.extend(
            attrs[attr_type]
        )

    flattened = sorted(
        flattened,
        key=lambda x: x["idx"],
    )

    names = [
        item["name"]
        for item in flattened
    ]

    coef_df = pd.DataFrame(
        {
            "feature": names,
            "coeficiente": coefs,
        }
    )

    numeric_name_map = {
        "numeric_scaled_0": "log_populacao",
        "numeric_scaled_1": (
            "vinculos_ativos_por_1000_habitantes"
        ),
        "numeric_scaled_2": (
            "proporcao_vinculos_estatutarios"
        ),
    }

    coef_df["feature"] = (
        coef_df["feature"]
        .replace(numeric_name_map)
    )

    coef_df["odds_ratio"] = (
        np.exp(
            coef_df["coeficiente"]
        )
    )

    coef_df["abs_coeficiente"] = (
        coef_df["coeficiente"]
        .abs()
    )

    return (
        coef_df
        .sort_values(
            "abs_coeficiente",
            ascending=False,
        )
        .reset_index(drop=True)
    )


## 16. Coeficientes do modelo final


In [ ]:
coef_df = extract_coefficients(
    final_model,
    df_train,
)

display(
    coef_df.head(30)
)


## 17. Principais fatores associados

São exibidos separadamente os fatores de maior associação positiva e negativa.


In [ ]:
print("Maiores associações positivas:")
display(
    coef_df
    .sort_values(
        "coeficiente",
        ascending=False,
    )
    .head(10)
)

print("Maiores associações negativas:")
display(
    coef_df
    .sort_values(
        "coeficiente",
        ascending=True,
    )
    .head(10)
)


## 18. Persistência da avaliação

As métricas comparativas dos modelos são gravadas em Gold.


In [ ]:
evaluation_rows = []

for result in [
    metrics_atual,
    metrics_enxuto,
]:
    for metrica in [
        "accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "balanced_accuracy",
        "macro_f1",
        "roc_auc",
    ]:
        evaluation_rows.append(
            (
                result["modelo"],
                2023,
                2024,
                metrica,
                float(result[metrica]),
                (
                    result["modelo"]
                    == FINAL_MODEL_NAME
                ),
            )
        )

df_evaluation = (
    spark.createDataFrame(
        evaluation_rows,
        [
            "modelo",
            "ano_treino",
            "ano_teste",
            "metrica",
            "valor",
            "modelo_final",
        ],
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp(),
    )
)

(
    df_evaluation
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        EVALUATION_TABLE
    )
)

display(
    spark.table(
        EVALUATION_TABLE
    )
)


## 19. Persistência dos coeficientes


In [ ]:
coef_spark = (
    spark.createDataFrame(
        coef_df[
            [
                "feature",
                "coeficiente",
                "odds_ratio",
                "abs_coeficiente",
            ]
        ]
    )
    .withColumn(
        "modelo",
        F.lit(FINAL_MODEL_NAME),
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp(),
    )
)

(
    coef_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        COEFFICIENTS_TABLE
    )
)

display(
    spark.table(
        COEFFICIENTS_TABLE
    )
    .orderBy(
        F.desc("abs_coeficiente")
    )
    .limit(30)
)


## Conclusão

A avaliação final permite responder três questões centrais:

### 1. O modelo generaliza para 2024?

A comparação entre validação de 2023 e teste temporal de 2024 mostra a estabilidade do modelo.

### 2. `regiao` deve permanecer?

O modelo com e sem região é comparado globalmente e especificamente nas UFs não vistas no treinamento.

### 3. Quais fatores mais influenciam a predição?

Os coeficientes e odds ratios da Regressão Logística fornecem interpretação direta e transparente.

### Uso executivo

O modelo não deve ser interpretado como causal nem como mecanismo para decisões individuais de alto impacto.

Seu uso mais adequado é **analítico e territorial**, apoiando:

- identificação de contextos com maior risco de não alfabetização;
- priorização de investigação em UFs e regiões;
- comparação de padrões territoriais;
- suporte à formulação de hipóteses e políticas educacionais.

### Próxima etapa

Com modelagem, avaliação e interpretabilidade concluídas, os próximos entregáveis são:

- documentação final no README;
- relatório executivo;
- visualizações para apresentação;
- roteiro do vídeo final.
